In [1]:
from __future__ import print_function, division
from optparse import OptionParser
from collections import defaultdict, OrderedDict
import concurrent.futures
import sys
import os
import rhalphalib as rl
import numpy as np
import pickle
import gzip
import json
from coffea import hist, processor
from coffea.util import load, save
from scipy import stats
import ROOT

rl.util.install_roofit_helpers()
rl.ParametericSample.PreferRooParametricHist = False

mT_binning_dict = {
    "2018": np.linspace(0,600,21),
    "2017": np.linspace(0,600,21),
    "2016": np.linspace(0,600,21)
}

Welcome to JupyROOT 6.18/00


In [11]:
def remap_histograms(hists):
    
    '''
    This method is used for mapping  the histograms in the .scaled file
    '''
    data_hists = {}
    bkg_hists = {}
    signal_hists = {}

    process = hist.Cat("process", "Process", sorting="placement")
    cats = ("process",)
    sig_map = OrderedDict()
    bkg_map = OrderedDict()
    data_map = OrderedDict()
    bkg_map["Hbb"] = ("Hbb*",)
    bkg_map["DY+jets"] = ("DY+*",)
    bkg_map["VV"] = (["WW", "WZ", "ZZ"],)
    bkg_map["ST"] = ("ST*",)
    bkg_map["TT"] = ("TT*",)
    bkg_map["W+jets"] = ("W+*",)
#     bkg_map["W+HF"] = ("W+HF",)
#     bkg_map["W+LF"] = ("W+LF",)
    bkg_map["Z+jets"] = ("Z+*",)
    bkg_map["QCD"] = ("QCD*",)
    data_map["SingleElectron"] = ("SingleElectron",)
    data_map["SingleMuon"] = ("SingleMuon",)
    data_map["EGamma"] = ("EGamma",)
    sig_map["Mphi-1995_Mchi-1000"] = ("*Mphi-1995_Mchi-1000*",)
    
#    processed signal 
    for signal in hists['sig']['template'].identifiers('process'):
        if 'Mphi' not in str(signal): continue
        sig_map[str(signal)] = (str(signal),)  ## signals

    for key in hists["data"].keys():
        bkg_hists[key] = hists["bkg"][key].group(cats, process, bkg_map)
        signal_hists[key] = hists["sig"][key].group(cats, process, sig_map)
        data_hists[key] = hists["data"][key].group(cats, process, data_map)

    print('initial mT_corrected binning',bkg_hists["template"].axis("mT_corrected").edges())


    bkg_hists["template"] = bkg_hists["template"].rebin("mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning))
    
    signal_hists["template"] = signal_hists["template"].rebin("mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning))
    data_hists["template"] = data_hists["template"].rebin("mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning))

    hists = {"bkg": bkg_hists, "data": data_hists, "sig": signal_hists}

    return hists


In [3]:
def template(dictionary, process, systematic, region, read_sumw2=False):
    histogram = dictionary[region].integrate("process", process)
    nominal, sumw2 = histogram.integrate("systematic", "nominal").values(sumw2=True)[()]
    zerobins = nominal <= 0.
    output = nominal
    
    if "nominal" not in systematic and "data" not in systematic:
    #print('Normalizing',systematic,'histogram of',process,'in region',region)
        output = histogram.integrate("systematic", systematic).values()[()]
        output[zerobins] = 1.
        output[~zerobins] /= nominal[~zerobins]
        output[np.isnan(output)] = 1.
        
    binning = (
        dictionary[region]
        .integrate("process", process)
        .integrate("systematic", systematic)
        .axis("mT")
        .edges()
    )
    if read_sumw2:
        return (output, binning, "mT", sumw2)
    return (output, binning, "mT")

In [9]:
!pwd

/cms/scratch/runiyal/new_coffea/decaf/analysis/Rishabh_NB/new_coffea_nbs/Combine related


In [13]:
year = '2018'
mT_corrected_binning = mT_binning_dict[year]
mTbins = np.array(mT_binning)
nmT = len(mTbins) - 1
hists = load("../../../hists/UL_lep_monotop_v1_test_" + year + ".scaled")
hists = remap_histograms(hists)
data_hists = hists["data"]
bkg_hists = hists["bkg"]
signal_hists = hists["sig"]

initial mT_corrected binning [  0.  30.  60.  90. 120. 150. 180. 210. 240. 270. 300. 330. 360. 390.
 420. 450. 480. 510. 540. 570. 600.]


In [22]:
###
# Preparing histograms for fit
##

data = {}
for r in data_hists["template"].identifiers("region"):
    data[str(r)] = data_hists["template"].integrate("region", r)

background = {}
for r in bkg_hists["template"].identifiers("region"):
    background[str(r)] = bkg_hists["template"].integrate("region", r)

signal = {}
for r in bkg_hists["template"].identifiers("region"):
    signal[str(r)] = signal_hists["template"].integrate("region", r)

{'sre': <Hist (process,systematic,mT_corrected) instance at 0x7fa8fad902b0>,
 'srm': <Hist (process,systematic,mT_corrected) instance at 0x7fa8fad903c8>,
 'ttbare': <Hist (process,systematic,mT_corrected) instance at 0x7fa8fad90470>,
 'ttbarm': <Hist (process,systematic,mT_corrected) instance at 0x7fa8fad90518>,
 'wjete': <Hist (process,systematic,mT_corrected) instance at 0x7fa8fad905c0>,
 'wjetm': <Hist (process,systematic,mT_corrected) instance at 0x7fa8fad90668>}

In [49]:
background['sre'].sum('process').sum('systematic').values()

{(): array([0.00000000e+00, 2.10877358e+05, 6.48343297e+05, 6.04468363e+05,
        6.16565120e+05, 4.87052933e+05, 2.81429958e+05, 1.45266031e+05,
        7.77396069e+04, 4.10414339e+04, 2.24940550e+04, 1.45189226e+04,
        8.23639357e+03, 4.69937151e+03, 3.47174406e+03, 2.28854791e+03,
        1.53409152e+03, 1.42941443e+03, 5.72489276e+02, 8.72617875e+02])}

In [44]:
background['sre'].axis("systematic").identifiers()

[<StringBin (JERDown) instance at 0x7fa8fa559240>,
 <StringBin (JERUp) instance at 0x7fa8fa5591d0>,
 <StringBin (JESDown) instance at 0x7fa8fa559128>,
 <StringBin (JESUp) instance at 0x7fa8fa559048>,
 <StringBin (METunclDown) instance at 0x7fa8fa5592b0>,
 <StringBin (METunclUp) instance at 0x7fa8fa559278>,
 <StringBin (btagDown) instance at 0x7fa8fa551c50>,
 <StringBin (btagUp) instance at 0x7fa8fa551c18>,
 <StringBin (ew1Down) instance at 0x7fa8fa559828>,
 <StringBin (ew1Up) instance at 0x7fa8fa559780>,
 <StringBin (ew2GDown) instance at 0x7fa8fa5598d0>,
 <StringBin (ew2GUp) instance at 0x7fa8fa559898>,
 <StringBin (ew2WDown) instance at 0x7fa8fa5599b0>,
 <StringBin (ew2WUp) instance at 0x7fa8fa559978>,
 <StringBin (ew2ZDown) instance at 0x7fa8fa559a58>,
 <StringBin (ew2ZUp) instance at 0x7fa8fa559a20>,
 <StringBin (ew3GDown) instance at 0x7fa8fa559b00>,
 <StringBin (ew3GUp) instance at 0x7fa8fa559ac8>,
 <StringBin (ew3WDown) instance at 0x7fa8fa559ba8>,
 <StringBin (ew3WUp) instance 